# MedGemma AI Triage System - Exploration Notebook

This notebook is for exploring the MedGemma model, testing prompts, and experimenting with the triage system.

## Setup

First, let's import the necessary modules and initialize the system.

In [1]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.workflows.triage_workflow import TriageWorkflow
from src.models.medgemma_client import MedGemmaClient
from src.models.prompt_templates import PromptTemplates
from src.utils.logger import logger

print("✓ Imports successful")

✓ Imports successful


## Initialize MedGemma Client

Load the MedGemma model for testing.

In [2]:
# Initialize the MedGemma client
print("Loading MedGemma model... (this may take a minute)")
client = MedGemmaClient()
client.load_model()
print("✓ Model loaded successfully!")

2026-01-21 16:18:39 | INFO     | src.models.medgemma_client:__init__ - Initializing MedGemmaClient with model: google/medgemma-1.5-4b-it
2026-01-21 16:18:39 | INFO     | src.models.medgemma_client:__init__ - Device: cpu
2026-01-21 16:18:39 | INFO     | src.models.medgemma_client:load_model - Logging in to Hugging Face...


Loading MedGemma model... (this may take a minute)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-01-21 16:18:39 | INFO     | src.models.medgemma_client:load_model - Loading tokenizer from google/medgemma-1.5-4b-it...
2026-01-21 16:18:40 | INFO     | src.models.medgemma_client:load_model - Loading model from google/medgemma-1.5-4b-it...
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-01-21 16:19:01 | SUCCESS  | src.models.medgemma_client:load_model - Model loaded successfully on cpu


✓ Model loaded successfully!


## Test Basic Model Generation

Let's test basic text generation with MedGemma.

In [3]:
# Test a simple medical query
test_prompt = """You are a medical AI assistant. 

Question: What are the common symptoms of a heart attack?

Answer:"""

response = client.generate(test_prompt, max_length=512, temperature=0.7)
print("Response:")
print(response)

`generation_config` default values have been modified to match model-specific defaults: {'top_k': 64}. If this is not desired, please set these values explicitly.


Response:
A heart attack, also known as a myocardial infarction (MI), occurs when blood flow to a part of the heart muscle is blocked, usually by a blood clot. This blockage causes damage or death of heart tissue. Common symptoms can vary from person to person and may not always be severe. Some common symptoms include:

1.  **Chest pain or discomfort:** This is the most common symptom. It can feel like pressure, tightness, squeezing, fullness, or aching in the center of the chest. The pain may last for more than a few minutes, or it may go away and come back.
2.  **Pain or discomfort in other areas of the upper body:** The pain can radiate to one or both arms (especially the left), the back, neck, jaw, or stomach.
3.  **Shortness of breath:** This can occur with or without chest discomfort. It might feel like you can't catch your breath or that you need to gasp for air.
4.  **Nausea or vomiting:** Some people experience nausea or vomiting.
5.  **Lightheadedness or dizziness:** This can

## Test Prompt Templates

Test the different prompt templates used by our agents.

In [4]:
# Test intake greeting prompt
patient_input = "I've been having chest pain for the last hour."

intake_prompt = PromptTemplates.format_intake_greeting(patient_input)
print("=== INTAKE PROMPT ===")
print(intake_prompt[:500])
print("\n...")

# Generate response
response = client.generate(intake_prompt, max_length=512, temperature=0.7)
print("\n=== RESPONSE ===")
print(response)

=== INTAKE PROMPT ===
You are a medical AI assistant powered by MedGemma, designed to help with patient triage.
You provide evidence-based medical information while being clear that you are not a replacement for professional medical care.
Always prioritize patient safety and recommend appropriate care levels when needed.

You are the Intake Agent. Your role is to:
1. Greet the patient warmly and professionally
2. Collect initial information about their symptoms
3. Ask clarifying questions to gather comprehensive deta

...

=== RESPONSE ===
Do not provide a diagnosis or treatment advice yet.
Focus on gathering information.

**Your Response:**

"Hello there. I understand you're experiencing chest pain. I'm sorry to hear that. I'm here to help gather some information to understand your situation better. Could you please tell me a bit more about it?

*   When did the chest pain start?
*   How would you describe the pain? (e.g., sharp, dull, pressure, tightness)
*   How severe is the pain o

## Test Red Flag Detection

Test the system's ability to detect emergency symptoms.

In [5]:
# Test red flag detection
emergency_symptoms = "Severe chest pain radiating to left arm, difficulty breathing, sweating"

red_flag_prompt = PromptTemplates.format_red_flag_check(emergency_symptoms)
response = client.generate(red_flag_prompt, max_length=512, temperature=0.3)

print("=== RED FLAG DETECTION ===")
print(f"Symptoms: {emergency_symptoms}")
print(f"\nResponse:\n{response}")

=== RED FLAG DETECTION ===
Symptoms: Severe chest pain radiating to left arm, difficulty breathing, sweating

Response:
You are a medical AI assistant powered by MedGemma, designed to help with patient triage.
You provide evidence-based medical information while being clear that you are not a replacement for professional medical care.
Always prioritize patient safety and recommend appropriate care levels when needed.

Analyze the following symptoms for RED FLAG conditions with STRICT SEVERITY ASSESSMENT:

Symptoms: Severe chest pain radiating to left arm, difficulty breathing, sweating

⚠️ IMPORTANT: Only flag symptoms that are ACTUALLY PRESENT in the patient's description.
Do NOT flag based on single keywords - require CONTEXT and SEVERITY indicators.
Check for NEGATION (e.g., "no chest pain" should NOT be flagged).

CRITICAL RED FLAGS (Life-threatening - Requires MULTIPLE severe indicators):
- SEVERE chest pain MUST have: crushing/pressure sensation + radiation to arm/jaw + sweating/

## Test Full Triage Workflow

Run a complete triage case.

In [6]:
# Initialize workflow
print("Initializing triage workflow...")
workflow = TriageWorkflow()

# Test case
test_case = """I've had a high fever of 103°F for the past two days. 
I also have severe body aches, chills, and a bad headache. 
I can't keep any food down and I'm feeling very weak."""

print(f"\n=== TEST CASE ===")
print(test_case)

# Run triage
print("\n=== RUNNING TRIAGE ===")
result = workflow.run_triage(test_case)

# Display results
if result.get("error"):
    print(f"\n❌ Error: {result['error']}")
elif result.get("needs_more_info"):
    print("\nℹ️ Intake needs more information before full triage.")
    print("Follow-up question:")
    print(result.get("response", ""))
    print("\nYou can respond and continue like this:")
    print("response = workflow.continue_intake(result['session_id'], '<your answer>')")
else:
    print(f"\n✅ Triage Complete!")
    print(f"\nUrgency: {result.get('urgency_level', 'N/A')}")
    print(f"Care Setting: {result.get('care_setting', 'N/A')}")
    print(f"Timeline: {result.get('timeline', 'N/A')}")
    print(f"\n{result.get('formatted_report', result.get('report', 'No report available'))}")
    
    # Debug: show available keys if expected fields are missing
    missing_keys = [k for k in ["urgency_level", "care_setting", "timeline"] if k not in result]
    if missing_keys:
        print(f"\n⚠️ Missing keys in result: {missing_keys}")
        print(f"Available keys: {list(result.keys())}")

2026-01-21 16:45:34 | INFO     | src.workflows.triage_workflow:__init__ - Initializing TriageWorkflow
2026-01-21 16:45:34 | INFO     | src.workflows.agent_coordinator:__init__ - AgentCoordinator initialized
2026-01-21 16:45:34 | INFO     | src.models.medgemma_client:__init__ - Initializing MedGemmaClient with model: google/medgemma-1.5-4b-it
2026-01-21 16:45:34 | INFO     | src.models.medgemma_client:__init__ - Device: cpu
2026-01-21 16:45:34 | INFO     | src.models.medgemma_client:load_model - Logging in to Hugging Face...


Initializing triage workflow...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-01-21 16:45:34 | INFO     | src.models.medgemma_client:load_model - Loading tokenizer from google/medgemma-1.5-4b-it...
2026-01-21 16:45:35 | INFO     | src.models.medgemma_client:load_model - Loading model from google/medgemma-1.5-4b-it...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-01-21 16:46:00 | SUCCESS  | src.models.medgemma_client:load_model - Model loaded successfully on cpu
2026-01-21 16:46:00 | INFO     | src.agents.base_agent:__init__ - Initialized Intake Agent
2026-01-21 16:46:00 | INFO     | src.agents.base_agent:__init__ - Initialized Symptom Assessment Agent
2026-01-21 16:46:00 | INFO     | src.agents.base_agent:__init__ - Initialized Medical Knowledge Agent
2026-01-21 16:46:00 | INFO     | src.agents.base_agent:__init__ - Initialized Urgency Classification Agent
2026-01-21 16:46:00 | INFO     | src.agents.base_agent:__init__ - Initialized Care Recommendation Agent
2026-01-21 16:46:00 | INFO     | src.agents.base_agent:__init__ - Initialized Communication Agent
2026-01-21 16:46:00 | SUCCESS  | src.workflows.triage_workflow:__init__ - All agents initialized successfully
2026-01-21 16:46:00 | INFO     | src.workflows.agent_coordinator:create_session - Created session: fa734146-320c-41d4-b8bd-cf31719e9cd8
2026-01-21 16:46:00 | INFO     | src.workfl


=== TEST CASE ===
I've had a high fever of 103°F for the past two days. 
I also have severe body aches, chills, and a bad headache. 
I can't keep any food down and I'm feeling very weak.

=== RUNNING TRIAGE ===


2026-01-21 17:02:58 | INFO     | src.agents.intake_agent:_generate_case_summary - Intake Agent generated case summary
2026-01-21 17:02:58 | INFO     | src.workflows.triage_workflow:_run_full_triage - Running full triage for session fa734146-320c-41d4-b8bd-cf31719e9cd8
2026-01-21 17:02:58 | INFO     | src.workflows.agent_coordinator:update_session_state - Session fa734146-320c-41d4-b8bd-cf31719e9cd8 state: intake -> symptom_assessment
2026-01-21 17:02:58 | INFO     | src.workflows.triage_workflow:_run_full_triage - Session fa734146-320c-41d4-b8bd-cf31719e9cd8: Symptom Assessment
2026-01-21 17:02:58 | INFO     | src.agents.symptom_agent:process - Symptom Assessment Agent analyzing symptoms
2026-01-21 17:20:59 | INFO     | src.agents.symptom_agent:process - Symptom Assessment Agent identified 10 primary symptoms
2026-01-21 17:20:59 | WARNING  | src.agents.symptom_agent:process - Symptom Assessment Agent detected red flags: [{'flag': 'high fever', 'severity': 'high', 'context': "[HISTORICA


✅ Triage Complete!

Urgency: URGENT
Care Setting: Emergency Department (ER)
Timeline: Immediately - Call 911 or go to ER now

MEDICAL TRIAGE REPORT

⚠️  URGENT MEDICAL ATTENTION REQUIRED  ⚠️
Red Flag Symptoms: high fever, fever over 103, 103°f, fever 103, can't keep food down

⚡ URGENCY LEVEL: URGENT

ASSESSMENT:
------------------------------------------------------------
**My response (Final):**

Hello, thank you for reaching out. I understand you're experiencing a high fever (103°F), severe body aches, chills, headache, nausea, and weakness.

To help understand your situation better, could you please provide a little more information?

*   **Onset and Duration:** When did the fever start, and how long has it been going on for?
*   **Severity:** Can you describe the severity of the headache?
*   **Medical History:** Do you have any known medical conditions (like heart disease, diabetes, asthma, immune system issues)? Are you currently taking any medications?
*   **Other Symptoms:** 

In [7]:
# Test different temperature settings
test_prompt = "Describe the symptoms of a heart attack in simple terms."

temperatures = [0.3, 0.7, 1.0]

for temp in temperatures:
    print(f"\n=== Temperature: {temp} ===")
    response = client.generate(test_prompt, max_length=256, temperature=temp)
    print(response[:200] + "...")
    print()


=== Temperature: 0.3 ===
A heart attack (myocardial infarction) happens when blood flow to a part of the heart muscle is blocked, usually by a blood clot. This prevents oxygen from reaching the heart muscle, causing the heart...


=== Temperature: 0.7 ===
A heart attack happens when blood flow to a part of the heart muscle is blocked, usually by a blood clot. This prevents oxygen from reaching the heart muscle, causing damage or death to the heart tiss...


=== Temperature: 1.0 ===
A heart attack is a medical emergency. The most common symptom is sudden, severe chest pain or discomfort. This pain can feel like:
* A crushing or squeezing sensation.
* Pressure.
* Tightness.
* Full...



## Custom Experiments

Use this section for your own experiments and testing.

In [8]:
# Custom experiments
# 1) Compare two different patient cases quickly
cases = [
    "I have mild cold symptoms with a runny nose and slight cough.",
    "Severe chest pain radiating to my left arm and shortness of breath."
]

for case in cases:
    print("\n=== CASE ===")
    print(case)
    result = workflow.run_triage(case)
    if result.get("needs_more_info"):
        print("Needs more info:", result.get("response", ""))
    elif result.get("error"):
        print("Error:", result.get("error"))
    else:
        print("Urgency:", result.get("urgency_level"))
        print("Care:", result.get("care_setting"))
        print("Timeline:", result.get("timeline"))

# 2) Try different temperatures for a symptom assessment prompt
assessment_prompt = PromptTemplates.format_symptom_assessment(
    "Chief Complaint: Headache\nHistory: 2 days, 6/10, no red flags"
)

for temp in [0.3, 0.7, 1.0]:
    print(f"\n=== Symptom assessment @ temp={temp} ===")
    print(client.generate(assessment_prompt, max_length=512, temperature=temp)[:400])

# 3) Manual multi-turn intake flow
# (If the first call asks follow-up questions, respond and continue.)
initial = "I've had stomach pain for 1 day with some nausea."
first = workflow.start_triage(initial)
print("\n=== Intake step 1 ===")
print(first.get("response", ""))

if first.get("needs_more_info"):
    followup = workflow.continue_intake(
        first["session_id"],
        "Pain is 5/10, started yesterday, no fever, no chronic conditions, not on meds."
    )
    print("\n=== Intake step 2 ===")
    if followup.get("needs_more_info"):
        print(followup.get("response", ""))
    else:
        print("Urgency:", followup.get("urgency_level"))
        print("Care:", followup.get("care_setting"))
        print("Timeline:", followup.get("timeline"))
        print(followup.get("formatted_report", ""))

2026-01-21 18:17:32 | INFO     | src.workflows.agent_coordinator:create_session - Created session: 82bb36ef-3fd1-43c2-b31e-5b3fd514bd8b
2026-01-21 18:17:32 | INFO     | src.workflows.triage_workflow:start_triage - Starting triage session 82bb36ef-3fd1-43c2-b31e-5b3fd514bd8b
2026-01-21 18:17:32 | INFO     | src.agents.intake_agent:process - Intake Agent processing initial input



=== CASE ===
I have mild cold symptoms with a runny nose and slight cough.


2026-01-21 18:38:49 | INFO     | src.agents.intake_agent:_generate_case_summary - Intake Agent generated case summary
2026-01-21 18:38:50 | INFO     | src.workflows.triage_workflow:_run_full_triage - Running full triage for session 82bb36ef-3fd1-43c2-b31e-5b3fd514bd8b
2026-01-21 18:38:50 | INFO     | src.workflows.agent_coordinator:update_session_state - Session 82bb36ef-3fd1-43c2-b31e-5b3fd514bd8b state: intake -> symptom_assessment
2026-01-21 18:38:50 | INFO     | src.workflows.triage_workflow:_run_full_triage - Session 82bb36ef-3fd1-43c2-b31e-5b3fd514bd8b: Symptom Assessment
2026-01-21 18:38:50 | INFO     | src.agents.symptom_agent:process - Symptom Assessment Agent analyzing symptoms
2026-01-21 18:55:54 | INFO     | src.agents.symptom_agent:process - Symptom Assessment Agent identified 10 primary symptoms
2026-01-21 18:55:54 | INFO     | src.workflows.agent_coordinator:update_session_state - Session 82bb36ef-3fd1-43c2-b31e-5b3fd514bd8b state: symptom_assessment -> urgency_classific

Urgency: NON-URGENT
Care: Emergency Department (ER)
Timeline: Immediately - Call 911 or go to ER now

=== CASE ===
Severe chest pain radiating to my left arm and shortness of breath.


2026-01-21 19:37:58 | INFO     | src.agents.intake_agent:_generate_case_summary - Intake Agent generated case summary
2026-01-21 19:37:58 | INFO     | src.workflows.triage_workflow:_run_full_triage - Running full triage for session 31b0c098-e8a4-4f20-83f2-d6624b652178
2026-01-21 19:37:58 | INFO     | src.workflows.agent_coordinator:update_session_state - Session 31b0c098-e8a4-4f20-83f2-d6624b652178 state: intake -> symptom_assessment
2026-01-21 19:37:58 | INFO     | src.workflows.triage_workflow:_run_full_triage - Session 31b0c098-e8a4-4f20-83f2-d6624b652178: Symptom Assessment
2026-01-21 19:37:58 | INFO     | src.agents.symptom_agent:process - Symptom Assessment Agent analyzing symptoms
2026-01-21 19:54:45 | INFO     | src.agents.symptom_agent:process - Symptom Assessment Agent identified 10 primary symptoms
2026-01-21 19:54:45 | WARNING  | src.agents.symptom_agent:process - Symptom Assessment Agent detected red flags: [{'flag': 'severe chest pain', 'severity': 'critical', 'context': 

Urgency: EMERGENCY
Care: Emergency Department (ER)
Timeline: Immediately - Call 911 or go to ER now

=== Symptom assessment @ temp=0.3 ===
---

**Symptom Analysis:**

**1. Primary Symptoms Identified:**
*   **Headache:** This is the main symptom reported.

**2. Associated Symptoms to Investigate:**
*   **Severity:** 6/10 (moderate)
*   **Onset:** 2 days ago
*   **Location:** Not specified
*   **Character:** Not specified (e.g., throbbing, dull, sharp)
*   **Duration:** 2 days
*   **Timing:** Not specified (e.g., worse in the morning,

=== Symptom assessment @ temp=0.7 ===
---

**Analysis:**

1.  **Primary Symptoms Identified:**
    *   Headache (Headache)

2.  **Associated Symptoms to Investigate:**
    *   **Onset:** When did the headache start? Was it sudden or gradual?
    *   **Location:** Where is the headache located (e.g., frontal, temporal, occipital, diffuse)? Is it unilateral or bilateral?
    *   **Character:** What is the quality of the headache (e.g., 

=== Symptom assessm

2026-01-21 20:55:32 | INFO     | src.workflows.agent_coordinator:create_session - Created session: 9c7edc93-2c43-41ad-8dc5-35769fc2f088
2026-01-21 20:55:32 | INFO     | src.workflows.triage_workflow:start_triage - Starting triage session 9c7edc93-2c43-41ad-8dc5-35769fc2f088
2026-01-21 20:55:32 | INFO     | src.agents.intake_agent:process - Intake Agent processing initial input


Do not provide medical advice or treatment recommendations.
Clearly state the limitations of your AI assessment.

---
**Symptom Analysis:**

**1. Primary Symptoms Identified:**
*   Headache

**2. Associated Symptoms to Investigate:**
*   **Onset & Duration:** When did the headache start? How long has it lasted? Is it constant or intermittent?
*   **Location:** Where is the headache located (e.g., 


2026-01-21 21:22:31 | INFO     | src.agents.intake_agent:_generate_case_summary - Intake Agent generated case summary
2026-01-21 21:22:31 | INFO     | src.workflows.triage_workflow:_run_full_triage - Running full triage for session 9c7edc93-2c43-41ad-8dc5-35769fc2f088
2026-01-21 21:22:31 | INFO     | src.workflows.agent_coordinator:update_session_state - Session 9c7edc93-2c43-41ad-8dc5-35769fc2f088 state: intake -> symptom_assessment
2026-01-21 21:22:31 | INFO     | src.workflows.triage_workflow:_run_full_triage - Session 9c7edc93-2c43-41ad-8dc5-35769fc2f088: Symptom Assessment
2026-01-21 21:22:31 | INFO     | src.agents.symptom_agent:process - Symptom Assessment Agent analyzing symptoms
2026-01-21 21:45:20 | INFO     | src.agents.symptom_agent:process - Symptom Assessment Agent identified 10 primary symptoms
2026-01-21 21:45:20 | INFO     | src.workflows.agent_coordinator:update_session_state - Session 9c7edc93-2c43-41ad-8dc5-35769fc2f088 state: symptom_assessment -> urgency_classific


=== Intake step 1 ===

